# Model A v2 — Frozen Backbone (controlled comparison test)

Purpose: isolate whether Model B's result is explained by the frozen-vs-unfrozen backbone mismatch, not by single-frame vs. temporal. This retrains Model A's single-frame architecture with the backbone **frozen** (matching Model B exactly) and the same training regime Model B uses (lr=1e-3, same dropout, same early stopping).

Reuses the same single-frame crops from Week 1 (`train_crops.h5` etc), no new extraction needed. This is a quick diagnostic, not the final Model A, the original fine-tuned Model A (F1=0.795) stays the reported baseline; this is an ablation to explain the Model A vs Model B gap.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/PIE_thesis'
CROPS_DIR = f'{DRIVE_ROOT}/crops'
RESULTS_DIR = f'{DRIVE_ROOT}/results'
CHECKPOINTS_DIR = f'{DRIVE_ROOT}/checkpoints'

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import torchvision.transforms as T
import numpy as np
import h5py
import random
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report
import pandas as pd

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print('Device:', device)


## 2. Dataset: reuses Week 1's single-frame crops (already extracted, no new work)

In [ ]:
IMG_SIZE = 224
normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

class PIECropsDataset(Dataset):
    def __init__(self, h5_path, train=False):
        self.h5_path = h5_path
        self.train = train
        with h5py.File(h5_path, 'r') as hf:
            self.length = hf['images'].shape[0]
        self._hf = None

    def _ensure_open(self):
        if self._hf is None:
            self._hf = h5py.File(self.h5_path, 'r')

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        self._ensure_open()
        img = self._hf['images'][idx]
        label = int(self._hf['labels'][idx])
        img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        if self.train and random.random() < 0.5:
            img = torch.flip(img, dims=[2])
        img = normalize(img)
        return img, label

train_ds = PIECropsDataset(f'{CROPS_DIR}/train_crops.h5', train=True)
val_ds = PIECropsDataset(f'{CROPS_DIR}/val_crops.h5')
test_ds = PIECropsDataset(f'{CROPS_DIR}/test_crops.h5')
print('Dataset sizes:', len(train_ds), len(val_ds), len(test_ds))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)


## 3. Class weights (same as before)

In [ ]:
with h5py.File(f'{CROPS_DIR}/train_crops.h5', 'r') as hf:
    train_labels = hf['labels'][:]

unique, counts = np.unique(train_labels, return_counts=True)
print('Train class distribution:', dict(zip(unique.tolist(), counts.tolist())))
class_weights = torch.tensor([len(train_labels) / c for c in counts], dtype=torch.float32).to(device)
print('Class weights:', class_weights)


## 4. Model: same MobileNetV2 single-frame architecture, backbone now FROZEN

This is the only structural change from the original Model A. Classifier head (dropout + linear) is identical and still trains. Learning rate and weight decay now match Model B's regime (1e-3 / 1e-4) since only a small classifier head is trainable, same reasoning Model B used.

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
for p in model.features.parameters():
    p.requires_grad = False
model.classifier[0] = nn.Dropout(p=0.5)
model.classifier[1] = nn.Linear(model.last_channel, 2)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
print(model)


## 5. Train (same regularization pattern: best checkpoint + early stopping)

In [ ]:
EPOCHS = 20
PATIENCE = 5

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            if train:
                optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            n += x.size(0)
            preds = out.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    return total_loss / max(n, 1), f1

history = []
best_val_f1 = 0.0
patience_counter = 0
for epoch in range(EPOCHS):
    train_loss, train_f1 = run_epoch(train_loader, train=True)
    val_loss, val_f1 = run_epoch(val_loader, train=False)
    print(f'Epoch {epoch+1}/{EPOCHS}  train_loss={train_loss:.4f} train_f1={train_f1:.4f}  val_loss={val_loss:.4f} val_f1={val_f1:.4f}')
    history.append({'epoch': epoch+1, 'train_loss': train_loss, 'train_f1': train_f1, 'val_loss': val_loss, 'val_f1': val_f1})
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), f'{CHECKPOINTS_DIR}/model_a_v2_frozen.pt')
        print(f'  New best val_f1={val_f1:.4f}, checkpoint saved')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1}, best val_f1={best_val_f1:.4f}')
            break

print('Training done, best val_f1:', best_val_f1)


## 6. Evaluate on held-out test set

In [ ]:
model.load_state_dict(torch.load(f'{CHECKPOINTS_DIR}/model_a_v2_frozen.pt'))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x)
        preds = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y.numpy())

f1 = f1_score(all_labels, all_preds, zero_division=0)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)

results = {'model': 'Model A v2 - Single-Frame, Frozen Backbone', 'f1': f1, 'precision': precision, 'recall': recall}
print(results)
print(classification_report(all_labels, all_preds, target_names=['not-crossing', 'crossing']))

metrics_df = pd.DataFrame([results])
metrics_df.to_csv(f'{RESULTS_DIR}/model_a_v2_test_metrics.csv', index=False)
history_df = pd.DataFrame(history)
history_df.to_csv(f'{RESULTS_DIR}/model_a_v2_training_history.csv', index=False)


## 7. What this tells us

Compare this F1 against:
- Original Model A (unfrozen backbone): F1 = 0.795
- Model B (frozen backbone + LSTM): F1 = 0.787

If this frozen-backbone Model A scores close to Model B (~0.787), the gap was mostly the backbone mismatch, not temporal information, meaning Model B's LSTM may be doing more than the raw F1 comparison suggested.

If this still scores close to the original 0.795, the backbone freezing wasn't the deciding factor, and the small Model A vs Model B gap is more likely about the (still-to-be-fixed) sampling window.